In [99]:
# ID3 Decision Tree

import math

In [100]:
# Start by defining the table data structure, in a dictionary of dictionaries


dataset = {
    '1': {
        'Stream': 'false',
        'Slope': 'steep',
        'Elevation': 'high',
        'Vegetation': 'chapparal',
          },
    '2': {
        'Stream': 'true',
        'Slope': 'moderate',
        'Elevation': 'low',
        'Vegetation': 'riparian',
          },
    '3': {
        'Stream': 'true',
        'Slope': 'steep',
        'Elevation': 'medium',
        'Vegetation': 'riparian',
          },
    '4': {
        'Stream': 'false',
        'Slope': 'steep',
        'Elevation': 'medium',
        'Vegetation': 'chapparal',
          },
    '5': {
        'Stream': 'false',
        'Slope': 'flat',
        'Elevation': 'high',
        'Vegetation': 'conifer',
          },
    '6': {
        'Stream': 'true',
        'Slope': 'steep',
        'Elevation': 'highest',
        'Vegetation': 'conifer',
          },
    '7': {
        'Stream': 'true',
        'Slope': 'steep',
        'Elevation': 'high',
        'Vegetation': 'chapparal',
          }
}


In [101]:
# Per attribute, define each one as a target and generate a tree

def entropy(data, target):
    # Calculate the entropy of a dataset
    # data is a dictionary of dictionaries
    # target is the attribute to be classified
    # entropy = - sum p(x) log2 p(x)

    # Count the number of instances in the dataset
    n = len(data)
    # Count the number of instances of each class
    class_count = {}
    for key in data:
        if data[key][target] in class_count:
            class_count[data[key][target]] += 1
        else:
            class_count[data[key][target]] = 1
    # Calculate the entropy
    entropy = 0
    for key in class_count:
        p = class_count[key]/n
        entropy -= p * math.log2(p)
    return entropy

In [102]:
# Test the entropy function
target_entropy = entropy(dataset, 'Vegetation')
print("Entropy of Vegetation: ", target_entropy)

Entropy of Vegetation:  1.5566567074628228


In [103]:
# Calculate the information gain of an attribute

def information_gain(data, target, attribute):
    # Calculate the information gain of an attribute
    # data is a dictionary of dictionaries
    # target is the attribute to be classified
    # attribute is the attribute to be tested
    # information gain = entropy(target) - sum p(x) entropy(x)

    # Calculate the entropy of the attribute
    n = len(data)
    attribute_entropy = 0
    value_count = {}
    for key in data:
        value = data[key][attribute]
        if value in value_count:
            value_count[value] += 1
        else:
            value_count[value] = 1
    for value in value_count:
        subset = {k: v for k, v in data.items() if v[attribute] == value}
        p = value_count[value] / n
        attribute_entropy += p * entropy(subset, target)
    # Calculate the information gain
    information_gain = entropy(data, target) - attribute_entropy
    return information_gain

In [104]:
# Test the information gain function
print("Stream")
print(information_gain(dataset, 'Stream', 'Vegetation'))
print("")
print("Slope")
print(information_gain(dataset, 'Slope', 'Vegetation'))
print("")
print("Elevation")
print(information_gain(dataset, 'Elevation', 'Vegetation'))

Stream
0.3059584928680418

Slope
0.5774062828523452

Elevation
0.877387064296613


In [110]:
# Do the ID3 algorithm based on the information gain

def ID3(data, target, attributes):
    # ID3 algorithm
    # data is a dictionary of dictionaries
    # target is the attribute to be classified
    # attributes is a list of attributes to be tested
    # returns a dictionary of dictionaries

    # Create a new node
    node = {}
    # Count the number of instances in the dataset
    n = len(data)
    # Count the number of instances of each class
    class_count = {}
    for key in data:
        if data[key][target] in class_count:
            class_count[data[key][target]] += 1
        else:
            class_count[data[key][target]] = 1
    # If all instances are of the same class, return the class
    if len(class_count) == 1:
        node['class'] = list(class_count.keys())[0]
        return node
    # If there are no attributes left, return the majority class
    if len(attributes) == 0:
        node['class'] = max(class_count, key=class_count.get)
        return node
    # Find the attribute with the highest information gain
    max_gain = 0
    best_attribute = ''
    for attribute in attributes:
        gain = information_gain(data, target, attribute)
        if gain > max_gain:
            max_gain = gain
            best_attribute = attribute
    # Create a new node with the best attribute
    node['attribute'] = best_attribute
    # Create a new node for each value of the best attribute
    node['values'] = {}
    for value in set([data[key][best_attribute] for key in data]):
        subset = {k: v for k, v in data.items() if v[best_attribute] == value}
        if len(subset) == 0:
            node['values'][value] = max(class_count, key=class_count.get)
        else:
            new_attributes = [a for a in attributes if a != best_attribute]
            node['values'][value] = ID3(subset, target, new_attributes)
    return node

# Test the ID3 algorithm
attributes = ['Elevation']
tree = ID3(dataset, 'Vegetation', attributes)
print(tree)

{'attribute': 'Elevation', 'values': {'low': {'class': 'riparian'}, 'highest': {'class': 'conifer'}, 'medium': {'class': 'riparian'}, 'high': {'class': 'chapparal'}}}
